In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
AUD = PROJECT_ROOT / "data" / "audit"
final = pd.read_csv(AUD / "news_dataset_reviewed.csv", encoding="utf-8-sig")
manifest = json.loads((AUD / "audit_manifest.json").read_text(encoding="utf-8"))
consistency = json.loads((AUD / "consistency_report.json").read_text(encoding="utf-8"))
sample = pd.read_csv(AUD / "label_sample_reviewed.csv", encoding="utf-8-sig")
label_audit = pd.read_csv(AUD / "label_audit_by_label.csv", encoding="utf-8-sig")
same_title = pd.read_csv(AUD / "same_title_groups.csv", encoding="utf-8-sig")
near = pd.read_csv(AUD / "near_duplicates.csv", encoding="utf-8-sig")
leak = pd.read_csv(AUD / "leakage_groups.csv", encoding="utf-8-sig")
date_comp = pd.read_csv(AUD / "date_comparison.csv", encoding="utf-8-sig")
print("Dataset:", final.shape)
print("Consistency:", consistency["checks_passed"], "/", consistency["checks_total"])

Dataset: (5898, 14)
Consistency: 32 / 32


In [2]:
assert len(final) == 5898
assert final.ID.nunique() == 5898
assert final.URL.nunique() == 5898
assert "Author" not in final.columns
assert final.Published_Date.isna().sum() == 0
assert set(final.Label) == {"Kinh tế", "Thế giới", "Thể thao", "Pháp luật",
                            "Giáo dục", "Sức khỏe", "Công nghệ", "Du lịch"}
assert (final.groupby("Label").size() >= 500).all()
print(final.groupby("Label").size().to_string())
print("\nTheo nguồn:")
print(final.groupby("Source").size().to_string())

Label
Công nghệ    664
Du lịch      746
Giáo dục     727
Kinh tế      740
Pháp luật    746
Sức khỏe     750
Thế giới     810
Thể thao     715

Theo nguồn:
Source
DanTri         780
ThanhNien      398
TuoiTre        399
VietnamNet    3152
VnExpress     1169


## 2. Kiểm định nhãn 80 bài

- 74/80 đúng trước hiệu chỉnh (92,5%).
- 4 bài đổi nhãn, 2 bài bị loại.
- 78/78 bài mẫu còn lại đúng sau hiệu chỉnh.
- 5.738 bài còn nhãn weak từ danh mục nguồn, chưa kiểm tra thủ công từng bài.

In [3]:
assert len(sample) == 80
assert (sample.Review_Status == "correct").sum() == 74
assert (sample.Review_Status == "corrected").sum() == 4
assert (sample.Review_Status == "drop").sum() == 2
print(sample.Review_Status.value_counts().to_string())
print("\nTheo lĩnh vực:")
print(label_audit.to_string(index=False))

Review_Status
correct      74
corrected     4
drop          2

Theo lĩnh vực:
    Label  N  Correct_Pre  Accuracy_Pre_Pct  Corrected  Dropped  Reviewed_Valid
Công nghệ 10            8              80.0          1        1               9
  Du lịch 10            9              90.0          0        1               9
 Giáo dục 10            9              90.0          1        0              10
  Kinh tế 10           10             100.0          0        0              10
Pháp luật 10           10             100.0          0        0              10
 Sức khỏe 10           10             100.0          0        0              10
 Thế giới 10            8              80.0          2        0              10
 Thể thao 10           10             100.0          0        0              10


In [4]:
print("Content length min/max/mean:", final.Content_Length.min(),
      final.Content_Length.max(), round(final.Content_Length.mean(), 1))
print("Bài 500–650 ký tự:", int(((final.Content_Length >= 500) & (final.Content_Length < 650)).sum()))
print("Bài >10.000 ký tự:", int((final.Content_Length > 10000).sum()))
assert final.Content_Length.min() >= 500

Content length min/max/mean: 501 34075 3279.3
Bài 500–650 ký tự: 13
Bài >10.000 ký tự: 30


In [5]:
assert same_title.groupby("Title_Group_Key").ngroups == 5 or same_title.Leakage_Group.nunique() == 5
assert len(near) == 17
assert final.groupby("Leakage_Group").Split.nunique().max() == 1
assert (final.groupby("Leakage_Group").size() > 1).sum() == 21
print("Split:", final.Split.value_counts().to_dict())
print("Số nhóm leakage >1 bài:", int((final.groupby("Leakage_Group").size() > 1).sum()))
print("\nCác nhóm trùng tiêu đề:")
print(same_title[["ID", "Source", "Label", "Title", "Leakage_Group"]].to_string(index=False))

Split: {'train': 4129, 'val': 886, 'test': 883}
Số nhóm leakage >1 bài: 21

Các nhóm trùng tiêu đề:
     ID     Source     Label                                                            Title Leakage_Group
C003280     DanTri  Giáo dục Vinschool phát triển tài năng học sinh theo lộ trình cá nhân hóa       LG01437
C003232  ThanhNien  Giáo dục         Thứ trưởng Bộ GD-ĐT nói về 'điều rất đẹp ở nghề dạy học'       LG01537
C003370 VietnamNet  Giáo dục           Thứ trưởng Bộ GD-ĐT nói về điều rất đẹp ở nghề dạy học       LG01537
C003013  VnExpress  Giáo dục Vinschool phát triển tài năng học sinh theo lộ trình cá nhân hóa       LG01437
C000215  ThanhNien   Kinh tế          Vingroup khởi công dự án nhà ở xã hội lớn nhất Việt Nam       LG02242
C000235  ThanhNien   Kinh tế          Vingroup khởi công dự án nhà ở xã hội lớn nhất Việt Nam       LG02242
C002681 VietnamNet Pháp luật                             Giang hồ mạng ‘Bình Trọc’ bị khởi tố       LG03097
C002376  VnExpress Pháp luật        

In [6]:
print(final.Date_Source.value_counts().to_string())
print("\nDate_Check:")
print(final.Date_Check.value_counts().to_string())
assert final.Published_Date.isna().sum() == 0
assert (date_comp.Date_Diff_Hours > 24).sum() == 313

Date_Source
jsonld:datePublished    5778
step3                     88
html_visible              32

Date_Check:
Date_Check
consistent_within_1h    4014
article_only_no_rss     1123
diff_over_24h            313
same_day_time_diff       276
diff_under_24h           172
